In [1]:
from pathlib import Path
import json
import polars as pl

In [2]:
bronze_dir = Path('../data/bronze/buildings')
silver_dir = Path('../data/silver/buildings')
silver_dir.mkdir(parents=True, exist_ok=True)

In [3]:
for file in sorted(bronze_dir.glob('*.parquet')):
    out = silver_dir / file.name

    lf = (
        pl.scan_parquet(file)
        .with_row_index('source_row_index')
        .with_columns(
            pl.col('upload_date').str.to_date('%Y-%m-%d'),
            pl.concat_str(
                [pl.col('source_file'), pl.lit(':'), pl.col('source_row_index').cast(pl.Utf8)]
            ).alias('silver_record_id'),
            pl.col('geometry').struct.field('type').alias('geometry_type'),
            pl.col('geometry')
            .map_elements(json.dumps, return_dtype=pl.Utf8)
            .alias('geometry_geojson'),
        )
        .filter(
            pl.col('geometry').is_not_null()
            & pl.col('geometry_type').is_in(['Polygon', 'MultiPolygon'])
        )
        .select(
          'country',
          'quadkey',
          'upload_date',
          'source_file',
          'source_row_index',
          'silver_record_id',
          'geometry',
          'geometry_type',
          'geometry_geojson',
        )
    )

    lf.sink_parquet(out)

In [4]:
# pl.read_parquet('../data/silver/buildings/Indonesia_132220333_2026-02-23.parquet')